# Hybrid Recommender Fix — Run These Cells In Order

In [1]:
# CELL 0 — Create the example_hybrid dataset from scratch
import sys
sys.argv = ['train.py']  # reset args

from pathlib import Path
from create_sample_dataset import create_sample_dataset
from data_loader import prepare_dataset
from embedding import load_embedding_model, load_or_create_embeddings
from collaborative import load_or_create_synthetic_interactions
from emotion_theme import add_emotion_theme_labels
import pandas as pd
import numpy as np

DATA_DIR = Path("data/example_hybrid")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 1. Build song catalog
print("Building song catalog...")
df = prepare_dataset()
df = df.reset_index(drop=True)
df["song_id"] = df.index.astype(str)
print(f"Songs: {len(df)}")

# 2. Song characteristics
song_chars = df[["song_id", "genre", "emotion", "theme",
                  "emotion_confidence", "theme_confidence", "word_count"]].copy()
song_chars.to_csv(DATA_DIR / "song_characteristics.csv", index=False)
print("✅ song_characteristics.csv")

# 3. Lyric embeddings
print("Loading embedding model (downloads once ~90MB)...")
model = load_embedding_model()
embeddings = load_or_create_embeddings(df, model, show_progress_bar=True)
emb_df = pd.DataFrame(embeddings, columns=[f"embedding_{i}" for i in range(embeddings.shape[1])])
emb_df.insert(0, "song_id", df["song_id"].values)
emb_df.to_csv(DATA_DIR / "lyric_embeddings.csv", index=False)
print(f"✅ lyric_embeddings.csv  shape={embeddings.shape}")

# 4. Synthetic interactions + user profiles (collaborative.py)
print("Generating synthetic interactions...")
interactions, profiles = load_or_create_synthetic_interactions(
    df,
    lyric_embeddings=embeddings,
    num_users=200,
    positives_per_user=20,
    seed=42,
    force_rebuild=True,
)
# Add play_count if missing
if "play_count" not in interactions.columns:
    interactions["play_count"] = interactions.get("preference_score", 1.0).clip(lower=0.1)
interactions.to_csv(DATA_DIR / "interactions.csv", index=False)
print(f"✅ interactions.csv  rows={len(interactions)}")

# 5. Basic user features (will be overwritten by regenerate_data.py next)
profiles.to_csv(DATA_DIR / "user_features.csv", index=False)
print(f"✅ user_features.csv  rows={len(profiles)}")

print("\nAll 4 CSVs created. Now run Cell 2 (regenerate_data.py).")

Building song catalog...
Songs: 1440
✅ song_characteristics.csv
Loading embedding model (downloads once ~90MB)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/45 [00:00<?, ?it/s]

✅ lyric_embeddings.csv  shape=(1440, 768)
Generating synthetic interactions...
✅ interactions.csv  rows=4000
✅ user_features.csv  rows=200

All 4 CSVs created. Now run Cell 2 (regenerate_data.py).


## CELL 1 — Check your CSV files exist

In [3]:
from pathlib import Path

DATA_DIR = Path('data/example_hybrid')

files = [
    'interactions.csv',
    'user_features.csv',
    'song_characteristics.csv',
    'lyric_embeddings.csv',
]

for f in files:
    p = DATA_DIR / f
    status = '✅ EXISTS' if p.exists() else '❌ MISSING'
    print(f'{status}  {p}')

✅ EXISTS  data/example_hybrid/interactions.csv
✅ EXISTS  data/example_hybrid/user_features.csv
✅ EXISTS  data/example_hybrid/song_characteristics.csv
✅ EXISTS  data/example_hybrid/lyric_embeddings.csv


## CELL 2 — Regenerate interactions.csv + user_features.csv
**Requires:** `song_characteristics.csv` and `lyric_embeddings.csv` must exist (✅ above).
This overwrites your old weak interactions with strong ones (20 neg/pos, 40% hard negatives).

In [ ]:
# Run regenerate_data.py as a module inline
exec(open('regenerate_data.py').read())

## CELL 3 — Verify new CSVs look correct

In [ ]:
import pandas as pd

interactions = pd.read_csv('data/example_hybrid/interactions.csv')
user_features = pd.read_csv('data/example_hybrid/user_features.csv')

print('=== interactions.csv ===')
print(f'Rows: {len(interactions)}')
print(f'Positives (interaction=1): {(interactions["interaction"]==1).sum()}')
print(f'Negatives (interaction=0): {(interactions["interaction"]==0).sum()}')
print(interactions.head(3))

print('\n=== user_features.csv ===')
print(f'Users: {len(user_features)}')
print(f'Features: {len(user_features.columns)}')
print(user_features.head(3))

## CELL 4 — Train
This runs `train.py` with all the fixed settings.
Takes ~5–15 min on CPU depending on dataset size.

In [5]:
import sys
sys.argv = [
    'train.py',
    '--data-dir', 'data/example_hybrid',
    '--epochs', '15',
    '--negatives-per-positive', '20',
    '--hard-negative-ratio', '0.40',
    '--lambda-rank', '2.0',
    '--lambda-point', '0.5',
    '--learning-rate', '5e-4',
    '--batch-size', '512',
    '--ranking-eval-candidates', '100',
    '--max-ranking-users', '50',
]

from trainer import main
main()

loaded hybrid dataset: users=200 songs=1440 train_positives=3200 validation_positives=800
epoch=01 train_total=1.2108 train_bpr=0.4759 val_recall@10=0.3900 val_ndcg@10=0.3267 lr=5.00e-04
epoch=02 train_total=0.1721 train_bpr=0.0490 val_recall@10=0.3850 val_ndcg@10=0.2750 lr=5.00e-04
epoch=03 train_total=0.0664 train_bpr=0.0143 val_recall@10=0.3650 val_ndcg@10=0.2880 lr=5.00e-04
epoch=04 train_total=0.0383 train_bpr=0.0071 val_recall@10=0.3650 val_ndcg@10=0.2799 lr=5.00e-04
epoch=05 train_total=0.0226 train_bpr=0.0033 val_recall@10=0.3500 val_ndcg@10=0.2633 lr=2.50e-04
epoch=06 train_total=0.0135 train_bpr=0.0017 val_recall@10=0.3500 val_ndcg@10=0.2652 lr=2.50e-04
epoch=07 train_total=0.0100 train_bpr=0.0011 val_recall@10=0.3250 val_ndcg@10=0.2508 lr=2.50e-04
epoch=08 train_total=0.0087 train_bpr=0.0011 val_recall@10=0.3250 val_ndcg@10=0.2551 lr=2.50e-04
epoch=09 train_total=0.0067 train_bpr=0.0005 val_recall@10=0.3400 val_ndcg@10=0.2537 lr=1.25e-04
epoch=10 train_total=0.0057 train_bpr

## CELL 5 — Evaluate (sampled, fast)

In [9]:
import sys
sys.argv = [
    'evaluate.py',
    '--k', '5', '10', '20',
    '--n-candidates', '100',
]

from evaluate import main
main()

Protocol: sampled(100)
@5: recall=0.1600  ndcg=0.1489  users=200
@10: recall=0.2900  ndcg=0.2203  users=200
@20: recall=0.4713  ndcg=0.2917  users=200
Saved to artifacts/hybrid_evaluation_metrics.json


## CELL 6 — Evaluate (full catalog, slow — run only after training converges)
Skip this until Cell 5 shows recall@10 > 0.30

In [11]:
import sys
sys.argv = [
    'evaluate.py',
    '--k', '5', '10', '20',
    '--full-catalog',
]

from evaluate import main
main()

Protocol: full_catalog
@5: recall=0.0225  ndcg=0.0200  users=200
@10: recall=0.0350  ndcg=0.0261  users=200
@20: recall=0.0587  ndcg=0.0354  users=200
Saved to artifacts/hybrid_evaluation_metrics.json


In [7]:
import sys
sys.argv = [
    'train.py',
    '--data-dir', 'data/example_hybrid',
    '--epochs', '5',          # sirf 5 epochs, epoch 1 best tha
    '--negatives-per-positive', '20',
    '--hard-negative-ratio', '0.40',
    '--lambda-rank', '2.0',
    '--lambda-point', '0.5',
    '--learning-rate', '5e-4',
    '--batch-size', '256',    # chhota batch chhote dataset ke liye better
    '--dropout', '0.3',       # regularization badhao overfitting rokne ke liye
    '--ranking-eval-candidates', '100',
    '--max-ranking-users', '50',
]

from importlib import reload
import trainer
reload(trainer)
from trainer import main
main()

loaded hybrid dataset: users=200 songs=1440 train_positives=3200 validation_positives=800
epoch=01 train_total=1.6817 train_bpr=0.6800 val_recall@10=0.3000 val_ndcg@10=0.2232 lr=5.00e-04
epoch=02 train_total=0.3715 train_bpr=0.1230 val_recall@10=0.2650 val_ndcg@10=0.2040 lr=5.00e-04
epoch=03 train_total=0.1636 train_bpr=0.0455 val_recall@10=0.2750 val_ndcg@10=0.2070 lr=5.00e-04
epoch=04 train_total=0.0951 train_bpr=0.0227 val_recall@10=0.3100 val_ndcg@10=0.2205 lr=5.00e-04
epoch=05 train_total=0.0618 train_bpr=0.0131 val_recall@10=0.2850 val_ndcg@10=0.2107 lr=2.50e-04
saved model to artifacts/hybrid_model.pt
saved metrics to artifacts/hybrid_training_metrics.json
best epoch: 1
best validation ndcg@10: 0.22321
